# 01 · 完整 RAG 问答应用（毕业项目）

> 学习目标：把 00~05 阶段学到的全部串起来，做出一个能运行的**私有知识问答应用**。

## 项目需求
- 知识来源：`data/人事管理流程.docx`（或换成你自己的文档）
- 完整链路：文档加载 → 切分 → 嵌入 → 向量库 → 检索 → 生成回答
- 进阶：加多轮记忆（03）、挂成 FastAPI/LangServe 服务（05-Deploy）

## 实现步骤清单
1. 加载文档（DocumentLoader）
2. 切分（TextSplitter）
3. 嵌入（Embeddings）
4. 入库（VectorStore）
5. 检索（Retriever）
6. 组装 RAG 链
7. （进阶）加记忆 + 部署

## Step 1-2：加载 + 切分

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 1：加载 docx
loader = Docx2txtLoader('./data/人事管理流程.docx')
docs = loader.load()
print('加载到的文档数：', len(docs))

# Step 2：切分
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print('切分后的块数：', len(chunks))

## Step 3-4：嵌入 + 入库

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Step 3：嵌入模型
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Step 4：入库
vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
print('向量库构建完成')

## Step 5-6：检索 + 组装 RAG 链

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 5：检索器
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

# Step 6：RAG 链 = 检索结果塞进 prompt -> 模型 -> 解析
prompt = ChatPromptTemplate.from_template(
    '根据以下资料回答问题，资料里没有的就说不知道。\n\n资料：\n{context}\n\n问题：{question}'
)
model = init_chat_model('deepseek-chat', model_provider='deepseek')

# ⭐ TODO: 把 retriever 检索到的 Document 拼成一段文本（retrieved_docs -> context 字符串）
# 提示：'\n\n'.join(doc.page_content for doc in docs)

def answer(question: str) -> str:
    retrieved_docs = retriever.invoke(question)
    context = '\n\n'.join(doc.page_content for doc in retrieved_docs)
    chain = prompt | model | StrOutputParser()
    return chain.invoke({'context': context, 'question': question})

print(answer('请假流程是什么？'))

## Step 7（进阶）：加记忆 + 部署

In [ ]:
# ⭐ 进阶方向（学完 03 和 05-Deploy 后回来补）
# 1. 多轮记忆：用 RedisChatMessageHistory 保存历史，每次把历史一起传给模型
# 2. 部署：参考 ../01-核心概念/deploy_service.py，用 LangServe 把 answer 挂成接口

# TODO: 动手改造

## 常见坑
- docx 加载器需要 `pip install docx2txt`（uv add docx2txt）
- `chunk_size` 不是越小越好：太小丢上下文，太大检索不准，500~1000 起步调
- 检索不到答案时要让模型回答"不知道"，避免胡编（prompt 里已约束）
- 官方 RAG 教程：https://python.langchain.com/docs/tutorials/rag/